# PC-RGNN: honest PCL graph-processing demonstration

This notebook verifies eight real PCL test-corpus files pinned by SHA-256. It distinguishes raw source files, derived normalized/subsampled views, and synthetic midpoint-completion points. The filename-to-class mapping is manual, and the corpus does not support a trustworthy class-balanced holdout, so this notebook intentionally reports no classification accuracy.

## Reproducibility and evaluation boundary

- Raw source: `data/pc_rgnn/*.pcd`, eight files, one `source_id` per file.
- Derived data: normalized and fixed-size views; these retain the source identifier.
- Synthetic data: midpoint interpolation used only to demonstrate graph completion.
- Evaluation: source-aware audit only. Augmented copies are never treated as independent scans.

In [ ]:
from pathlib import Path
import hashlib, json
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'data' / 'pc_rgnn').exists():
    ROOT = ROOT.parent
MANIFEST_PATH = ROOT / 'data' / 'pc_rgnn' / 'provenance.json'
manifest = json.loads(MANIFEST_PATH.read_text())
DATA_DIR = ROOT / 'data' / 'pc_rgnn'
print('project root:', ROOT)
print('pinned revision:', manifest['pinned_revision'])

In [ ]:
def sha256(path):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

def read_ascii_pcd(path):
    header = []
    with path.open('rb') as handle:
        for raw_line in handle:
            line = raw_line.decode('ascii').strip()
            header.append(line)
            if line.upper().startswith('DATA'):
                break
    fields = next(line.split()[1:] for line in header if line.upper().startswith('FIELDS'))
    data_line = next(line for line in header if line.upper().startswith('DATA'))
    if data_line.lower() != 'data ascii':
        raise ValueError(f'{path.name}: expected DATA ascii')
    values = np.loadtxt(path, skiprows=len(header), ndmin=2)
    xyz = values[:, [fields.index(axis) for axis in ('x', 'y', 'z')]].astype(np.float32)
    if not np.isfinite(xyz).all():
        raise ValueError(f'{path.name}: non-finite XYZ')
    return xyz

records = []
for item in manifest['files']:
    path = DATA_DIR / item['file']
    assert path.exists(), path
    assert path.stat().st_size == item['bytes']
    assert sha256(path) == item['sha256']
    points = read_ascii_pcd(path)
    records.append({'source_id': item['file'], 'class': item['class'], 'points': points, 'path': path})
print(f'verified {len(records)} raw PCL sources')
for row in records:
    print(row['source_id'], row['class'], row['points'].shape[0], 'points')

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
names = [row['source_id'] for row in records]
counts = [row['points'].shape[0] for row in records]
colors = ['#2563eb' if row['class'] == 'bunny' else '#64748b' for row in records]
ax.bar(names, counts, color=colors)
ax.set_title('Raw PCL source inventory (8 independent files; no augmentations)')
ax.set_ylabel('points in source file')
ax.tick_params(axis='x', rotation=45)
fig.tight_layout()
(ROOT / 'docs' / 'plots').mkdir(parents=True, exist_ok=True)
fig.savefig(ROOT / 'docs' / 'plots' / 'raw_inventory.png', dpi=160)
plt.show()

In [ ]:
def normalize(points):
    centered = points - points.mean(axis=0, keepdims=True)
    scale = np.linalg.norm(centered, axis=1).max()
    return centered / max(scale, 1e-8)

def sample_fixed(points, n=256, seed=7):
    rng = np.random.default_rng(seed)
    indices = rng.choice(len(points), size=n, replace=len(points) < n)
    return points[indices]

def graph_complete(points, k=8):
    try:
        from scipy.spatial import cKDTree
    except ImportError:
        raise RuntimeError('scipy is required for graph completion')
    distances, neighbours = cKDTree(points).query(points, k=min(k + 1, len(points)))
    neighbours = np.atleast_2d(neighbours)
    source = points
    midpoints = (source + source[neighbours[:, 1]]) / 2.0
    return np.concatenate([source, midpoints], axis=0)

raw = normalize(records[0]['points'])
observed = sample_fixed(raw, n=min(256, len(raw)))
completed = graph_complete(observed, k=6)
print('source:', records[0]['source_id'])
print('observed derived view:', observed.shape, '| synthetic completion:', completed.shape)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].scatter(observed[:, 0], observed[:, 1], s=5, alpha=.7)
axes[0].set_title('Derived normalized/subsampled view')
axes[1].scatter(completed[:, 0], completed[:, 1], s=5, alpha=.55)
axes[1].set_title('Derived + synthetic midpoint points')
for axis in axes:
    axis.set_aspect('equal')
fig.suptitle('PCL graph completion demonstration; synthetic points are not new scans')
fig.tight_layout()
fig.savefig(ROOT / 'docs' / 'plots' / 'completion_before_after.png', dpi=160)
plt.show()

In [ ]:
class_counts = {}
for row in records:
    class_counts[row['class']] = class_counts.get(row['class'], 0) + 1
print('source counts by manual filename label:', class_counts)
source_aware_evaluation_supported = all(count >= 2 for count in class_counts.values())
assert len({row['source_id'] for row in records}) == len(records)
if not source_aware_evaluation_supported:
    print('CLASSIFICATION METRICS DISABLED: at least one manual class has fewer than two independent source files.')
    print('This prevents a misleading augmented-copy train/validation claim.')
else:
    raise RuntimeError('Update this notebook with a source-aware split before reporting classification.')

In [ ]:
try:
    import torch
    from torch import nn

    def knn_indices(points, k):
        from scipy.spatial import cKDTree
        return cKDTree(points).query(points, k=k + 1)[1][:, 1:]

    class EdgeAttentionBlock(nn.Module):
        def __init__(self, channels):
            super().__init__()
            self.query = nn.Linear(channels, channels, bias=False)
            self.key = nn.Linear(channels, channels, bias=False)
            self.value = nn.Linear(channels, channels, bias=False)
            self.out = nn.Linear(channels, channels)

        def forward(self, x, neighbours):
            batch, points, channels = x.shape
            gather = neighbours[None, :, :, None].expand(batch, -1, -1, channels)
            keys = torch.gather(x[:, :, None, :].expand(-1, -1, neighbours.shape[1], -1), 1, gather)
            values = torch.gather(x[:, :, None, :].expand(-1, -1, neighbours.shape[1], -1), 1, gather)
            scores = (self.query(x)[:, :, None, :] * self.key(keys)).sum(-1) / channels**0.5
            weights = scores.softmax(-1)
            return x + self.out((weights[..., None] * self.value(values)).sum(2)), weights

    class PCRGNNEncoder(nn.Module):
        def __init__(self, channels=32):
            super().__init__()
            self.input = nn.Linear(3, channels)
            self.local = EdgeAttentionBlock(channels)
            self.context = EdgeAttentionBlock(channels)

        def forward(self, points, local_idx, context_idx):
            x = self.input(points)
            x, local_attention = self.local(x, local_idx)
            x, context_attention = self.context(x, context_idx)
            return x.mean(1), local_attention, context_attention

    demo = sample_fixed(normalize(records[0]['points']), n=64)
    local_idx = torch.tensor(knn_indices(demo, 6), dtype=torch.long)
    context_idx = torch.tensor(knn_indices(demo, 12), dtype=torch.long)
    embedding, local_attention, context_attention = PCRGNNEncoder()(torch.tensor(demo[None], dtype=torch.float32), local_idx, context_idx)
    print('attention encoder output:', tuple(embedding.shape), 'local weights:', tuple(local_attention.shape), 'context weights:', tuple(context_attention.shape))
except ImportError as exc:
    print('PyTorch/scipy encoder skipped:', exc)

## Conclusion

The PCL files pass the pinned integrity checks and the graph/attention forward pass is executable when the optional ML dependencies are installed. The completion points are synthetic and the manual labels are not official annotations. Because the source-aware class-balanced holdout is not supported by this corpus, the old augmented validation accuracy is intentionally not reproduced or presented as evidence of generalization.